# Multi Layer Perceptron - Architecture and normalization

This notebook experiments with how mlp architecture and data normalization influences the classification results

In [ ]:
import sys

sys.path.append("..")

from ucimlrepo import fetch_ucirepo
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
heart_disease = fetch_ucirepo(id=45)

X_raw: pd.DataFrame = heart_disease.data.features
y_raw: pd.Series = (heart_disease.data.targets["num"] > 0).astype(int)

In [ ]:
mask = ~np.asarray(pd.isna(X_raw).any(axis=1))

X = X_raw[mask]
y = y_raw[mask]

In [ ]:
from models.mlp import MultiLayerPerceptron
from models.activation_functions import ReLu
from models.loss_functions import LogLoss
from models.optimizer import SGD
from models.weight_initializer import NormalInitializer

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler

NUM_COLS = ["age", "trestbps", "chol", "thalach", "oldpeak", "ca"]

prep = ColumnTransformer(
    transformers=[
        ("stand", StandardScaler(), NUM_COLS),
    ],
    remainder="passthrough",
)

In [ ]:
from sklearn.base import clone
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

X_train, X_test, y_train, y_test = train_test_split(
    X, y.to_numpy(), test_size=0.2, stratify=y, random_state=42
)

hidden_layer_architectures = [
    [13],
    [67],
    [8],
    [13, 13, 13],
    [32, 16, 8, 4, 2],
    [1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
    [2, 4, 8, 16, 32],
    [24, 16],
    [8, 24],
]

results = []

for hidden_layer_sizes in hidden_layer_architectures:
    for use_standard_scaler in [True, False]:
        pipe = Pipeline(
            [
                ("prep", clone(prep) if use_standard_scaler else "passthrough"),
                (
                    "clf",
                    MultiLayerPerceptron(
                        hidden_layer_sizes=hidden_layer_sizes,
                        activation_function=ReLu(),
                        loss_function=LogLoss(),
                        weight_initializer=NormalInitializer(0.1, random_state=42),
                        optimizer=SGD(),
                        random_state=42,
                    ),
                ),
            ]
        )

        pipe.fit(X_train, y_train)
        y_pred = pipe.predict(X_test)

        metrics = {
            "accuracy": round(accuracy_score(y_test, y_pred), 2),
            "f1": round(f1_score(y_test, y_pred, zero_division=0), 2),
            "precision": round(precision_score(y_test, y_pred, zero_division=0), 2),
            "recall": round(recall_score(y_test, y_pred, zero_division=0), 2),
        }

        print(
            f"hidden_layer_sizes={hidden_layer_sizes}, "
            f"standard_scaler={use_standard_scaler} | "
            f"accuracy={metrics['accuracy']:.2f}, "
            f"f1={metrics['f1']:.2f}, "
            f"precision={metrics['precision']:.2f}, "
            f"recall={metrics['recall']:.2f}"
        )

        pipe.named_steps["clf"].plot_history()
        plt.show()

        results.append(
            {
                "hidden_layer_sizes": str(hidden_layer_sizes),
                "standard_scaler": use_standard_scaler,
                **metrics,
            }
        )

results_df = pd.DataFrame(results)
results_df.sort_values(
    by=["hidden_layer_sizes", "standard_scaler"],
    ascending=[True, False],
).reset_index(drop=True)